[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhmk/hocmaytrongtaichinh/blob/main/Module_7/L4.ipynb)

## HỌC MÁY TRONG TÀI CHÍNH
MODULE 7 | BÀI 4


---

# **CHIẾN LƯỢC GIAO DỊCH BITCOIN**


|  |  |
|:---|:---|
|**Thời gian đọc** |  30 phút |
|**Kiến thức nền** | Tinh chỉnh siêu tham số, học máy có giám sát, các thuật toán phân loại  |
|**Từ khóa** |tối ưu hóa Bayes, tín hiệu  |


---

In [ ]:
# Chuẩn bị môi trường trên Colab (chạy ô này một lần trước khi chạy các ô khác)
%pip install -q bayesian-optimization yfinance
import os, urllib.request, urllib.parse

BASE = "https://raw.githubusercontent.com/thanhmk/hocmaytrongtaichinh/main/Module_7/"

*Ở bài trước, ta đã bàn về các hàm mục tiêu và mở rộng hiểu biết về tinh chỉnh siêu tham số. Trong bài này, ta sẽ huấn luyện các thuật toán phân loại trên bài toán chiến lược giao dịch Bitcoin và cải thiện hiệu suất của một mô hình học máy bằng cách tinh chỉnh tham số của nó với tìm kiếm lưới và tối ưu hóa Bayes.*

## **1. Giới thiệu**

Trong bài toán này, ta muốn dự báo thời điểm nên mua hoặc bán Bitcoin. Ta bắt đầu bằng việc định nghĩa một *tín hiệu mua hoặc bán*, biểu diễn bằng 1 hoặc 0. Ta có tín hiệu này bằng cách so sánh xu hướng giá ngắn hạn và dài hạn: nếu đường trung bình động ngắn hạn lớn hơn đường trung bình động dài hạn thì ta mua bitcoin, ngược lại thì bán. Do đó đây là một bài toán phân loại, trong đó ta quan tâm đến việc đoán đúng hướng đi của Bitcoin. Ta sẽ chia tập dữ liệu thành tập huấn luyện và tập kiểm định theo tỷ lệ 4:1.

## **2. Nạp thư viện hỗ trợ và dữ liệu**

### **2.1 Nạp thư viện hỗ trợ**

Ta nạp các thư viện Python giúp ta đạt mục tiêu.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set()

In [ ]:
from time import time

import joblib

# display all columns
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)


from bayes_opt import BayesianOptimization

# Libraries for Deep Learning Models
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression

# needed for `HistGradientBoostingClassifier`
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    StratifiedKFold,
    cross_val_score,
    cross_validate,
    train_test_split,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

# from `keras.optimizers` import SGD

### **2.2 Nạp dữ liệu**

Ta lấy dữ liệu giá Bitcoin từ Yahoo Finance trong năm năm gần nhất.

In [ ]:
# To fetch financial data
import yfinance as yf

# Set the ticker as 'BTC-USD'
BTC_Ticker = yf.Ticker("BTC-USD")
BTC_Data = BTC_Ticker.history(period="5y")

In [ ]:
# BTC_Ticker = yf.Ticker("BTC-USD")
# BTC_Data = BTC_Ticker.history(period="max")
BTC_Data.head()

### **3. Phân tích dữ liệu khám phá (EDA)**

Giai đoạn này khám phá dữ liệu để giúp ta thấy các xu hướng trong dữ liệu.

In [ ]:
BTC_Data.shape

In [ ]:
BTC_Data.info()

In [ ]:
BTC_Data.describe()

In [ ]:
# Checking for any null values and removing the null values'''
print("Null Values =", BTC_Data.isnull().values.any())

## **4. Chuẩn bị dữ liệu**

Ta bắt đầu bằng việc tạo biến mục tiêu, tức là tín hiệu mua hoặc bán. Biến mục tiêu cấu thành chiến lược giao dịch của ta. Khi đường trung bình động ngắn hạn vượt lên trên đường trung bình động dài hạn, đó là chỉ báo nên mua, và ngược lại cũng đúng. Đọc thêm tại đây: [Picking buy-sell signals: A practitioner's perspective on key technical indicators for selected Indian firms.](https://sciendo.com/article/10.2478/sbe-2019-0054)

In [ ]:
# Create short simple moving average over the short window
BTC_Data["short_moving_avg"] = (
    BTC_Data["Close"].rolling(window=10, min_periods=1, center=False).mean()
)

# Create long simple moving average over the long window
BTC_Data["long_maving_avg"] = (
    BTC_Data["Close"].rolling(window=60, min_periods=1, center=False).mean()
)

# Create signals
BTC_Data["signal"] = np.where(
    BTC_Data["short_moving_avg"] > BTC_Data["long_maving_avg"], 1.0, 0.0
)

In [ ]:
BTC_Data.head()

### **4.1 Kỹ thuật đặc trưng**

Bây giờ ta tạo thêm các đặc trưng trong tập dữ liệu để giúp cải thiện hiệu suất mô hình. Các đặc trưng bổ sung là:
1. Trung bình động hàm mũ (Exponential Moving Average): cho ta xu hướng giá của dữ liệu.
2. Chỉ số sức mạnh tương đối (RSI): đo mức thay đổi giá trong một khung thời gian gần đây.
3. Tốc độ thay đổi (Rate of Change): đo phần trăm thay đổi giữa giá hiện tại của cổ phiếu và các mức giá trước đó.
4. Dao động ngẫu nhiên (Stochastic Oscillator): so sánh giá đóng cửa hiện tại của cổ phiếu với các giá đóng cửa trước đó.

In [ ]:
# calculation of exponential moving average


def EMA(df, n):
    EMA = pd.Series(df["Close"].ewm(span=n, min_periods=n).mean(), name="EMA_" + str(n))
    return EMA


BTC_Data["EMA10"] = EMA(BTC_Data, 10)
BTC_Data["EMA30"] = EMA(BTC_Data, 30)
BTC_Data["EMA200"] = EMA(BTC_Data, 200)
BTC_Data.head()


# calculation of rate of change
def RC(df, n):
    M = df.diff(n - 1)
    N = df.shift(n - 1)
    RC = pd.Series(((M / N) * 100), name="RC_" + str(n))
    return RC


BTC_Data["RC10"] = RC(BTC_Data["Close"], 10)
BTC_Data["RC30"] = RC(BTC_Data["Close"], 30)


# Calculation of price momentum
def MOM(df, n):
    MOM = pd.Series(df.diff(n), name="Momentum_" + str(n))
    return MOM


BTC_Data["MOM10"] = MOM(BTC_Data["Close"], 10)
BTC_Data["MOM30"] = MOM(BTC_Data["Close"], 30)


# calculation of relative strength index
def RSI(series, period):
    delta = series.diff().dropna()
    u = delta * 0
    d = u.copy()
    u[delta > 0] = delta[delta > 0]
    d[delta < 0] = -delta[delta < 0]
    u[u.index[period - 1]] = np.mean(u[:period])  # first value is sum of avg gains
    u = u.drop(u.index[: (period - 1)])
    d[d.index[period - 1]] = np.mean(d[:period])  # first value is sum of avg losses
    d = d.drop(d.index[: (period - 1)])
    rs = (
        u.ewm(com=period - 1, adjust=False).mean()
        / d.ewm(com=period - 1, adjust=False).mean()
    )
    return 100 - 100 / (1 + rs)


BTC_Data["RSI10"] = RSI(BTC_Data["Close"], 10)
BTC_Data["RSI30"] = RSI(BTC_Data["Close"], 30)
BTC_Data["RSI200"] = RSI(BTC_Data["Close"], 200)

# calculation of stochastic oscillator.


def STOK(close, low, high, n):
    STOK = (
        (close - low.rolling(n).min()) / (high.rolling(n).max() - low.rolling(n).min())
    ) * 100
    return STOK


def STOD(close, low, high, n):
    STOK = (
        (close - low.rolling(n).min()) / (high.rolling(n).max() - low.rolling(n).min())
    ) * 100
    STOD = STOK.rolling(3).mean()
    return STOD


BTC_Data["%K10"] = STOK(BTC_Data["Close"], BTC_Data["Low"], BTC_Data["High"], 10)
BTC_Data["%D10"] = STOD(BTC_Data["Close"], BTC_Data["Low"], BTC_Data["High"], 10)
BTC_Data["%K30"] = STOK(BTC_Data["Close"], BTC_Data["Low"], BTC_Data["High"], 30)
BTC_Data["%D30"] = STOD(BTC_Data["Close"], BTC_Data["Low"], BTC_Data["High"], 30)
BTC_Data["%K200"] = STOK(BTC_Data["Close"], BTC_Data["Low"], BTC_Data["High"], 200)
BTC_Data["%D200"] = STOD(BTC_Data["Close"], BTC_Data["Low"], BTC_Data["High"], 200)

Ta có thể hiển thị dataframe để xem các đặc trưng vừa tạo.

In [ ]:
BTC_Data.tail()

Trong ô tiếp theo, ta tính trung bình động.

In [ ]:
def MA(df, n):
    MA = pd.Series(df["Close"].rolling(n, min_periods=n).mean(), name="MA_" + str(n))
    return MA


BTC_Data["MA21"] = MA(BTC_Data, 10)
BTC_Data["MA63"] = MA(BTC_Data, 30)
BTC_Data["MA252"] = MA(BTC_Data, 200)
BTC_Data.tail()

Bây giờ ta có thể loại các đặc trưng không liên quan đến nhiệm vụ dự báo, như các giá trị `High`, `Low` và `Open` của Bitcoin. Vì đã định nghĩa xong tín hiệu giao dịch, ta bỏ các đặc trưng trung bình động ngắn hạn và dài hạn.

In [ ]:
# excluding columns that are not needed for our prediction.

BTC_Data = BTC_Data.drop(
    [
        "High",
        "Low",
        "Open",
        "short_moving_avg",
        "long_maving_avg",
        "Dividends",
        "Stock Splits",
    ],
    axis=1,
)

In [ ]:
BTC_Data = BTC_Data.dropna(axis=0)

Bây giờ ta có thể trực quan hóa xu hướng giá Bitcoin trong bốn năm như hình bên dưới.

In [ ]:
BTC_Data[["Close"]].plot(grid=True)
plt.suptitle(
    "Fig. 1: Bitcoin Price trend", fontweight="bold", horizontalalignment="right"
)
plt.show()

Trong ô bên dưới, ta hiển thị phân phối của các đặc trưng, giúp người xây dựng mô hình quyết định cách xử lý từng đặc trưng trong tập dữ liệu dự báo.

In [ ]:
# histograms
BTC_Data.hist(sharex=False, sharey=False, xlabelsize=1, ylabelsize=1, figsize=(12, 12))
plt.suptitle(
    "Fig. 2: Variable Distribution", fontweight="bold", horizontalalignment="right"
)
plt.show()

Như đã nêu ở phần giới thiệu, ta quan tâm đến việc dự báo thời điểm mua hoặc bán Bitcoin. Ta vẽ biểu đồ tròn để trực quan hóa tỷ lệ tín hiệu bán so với tín hiệu mua trong tập dữ liệu.

In [ ]:
fig = plt.figure()
plot = (
    BTC_Data.groupby("signal")
    .size()
    .plot(kind="pie", figsize=(6, 6), autopct="%1.1f%%", y="Signal")
)
plt.suptitle(
    "Fig. 3: A Pie Chart of buy and sell signal",
    fontweight="bold",
    horizontalalignment="right",
)
plt.legend(["sell", "buy"])
plt.show()

Số đếm các tín hiệu này cũng có thể xem bên dưới.

In [ ]:
BTC_Data.groupby(["signal"]).size()

Bây giờ ta có thể xem mối quan hệ giữa các biến bằng ma trận tương quan của lợi suất.

In [ ]:
# correlation
BTC_data_returns = BTC_Data.shift(1)
correlation = BTC_data_returns.corr()
plt.figure(figsize=(15, 15))
plt.title("Correlation Matrix")
sns.heatmap(correlation, vmax=1, square=True, annot=True, cmap="RdBu")
plt.suptitle(
    "Fig. 4: Correlation Matrix of BITCOIN return",
    fontweight="bold",
    horizontalalignment="right",
)
plt.show()

## **5. Xây dựng mô hình**

Khi đã có tập dữ liệu, ta chia nó thành tập huấn luyện và tập giữ lại (holdout); trong trường hợp này, tập kiểm tra sẽ dùng để kiểm thử ngược (backtest) mô hình.

### **5.1 Chia tập huấn luyện và kiểm tra**

Ta bắt đầu quá trình xây dựng mô hình bằng việc chia tập dữ liệu thành tập huấn luyện và tập kiểm tra theo tỷ lệ 80:20.

In [ ]:
def get_holdout_set(target, features, period=6):
    idx = pd.IndexSlice
    label = target.name
    dates = np.sort(y.index.get_level_values("Date").unique())
    cv_start, cv_end = dates[0], dates[-period - 2]
    holdout_start, holdout_end = dates[-period - 1], dates[-1]

    df = features.join(target.to_frame())
    train = df.loc[idx[:, cv_start:cv_end], :]
    y_train, X_train = train[label], train.drop(label, axis=1)

    test = df.loc[idx[:, holdout_start:holdout_end], :]
    y_test, X_test = test[label], test.drop(label, axis=1)
    return y_train, X_train, y_test, X_test

In [ ]:
subset_validation = BTC_Data[1301:]

In [ ]:
subset_validation.shape

In [ ]:
# split out validation dataset for the end
subset_dataset = BTC_Data[:1300]
y = subset_dataset["signal"]
X = subset_dataset.loc[:, BTC_Data.columns != "signal"]
X.index = pd.to_datetime(X.index)
validation_size = 0.2
seed = 1
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=validation_size, random_state=1
)

In [ ]:
subset_dataset.shape

Ta tạo một tập **kiểm định** gồm phần dữ liệu còn lại mà ta chưa chọn vào tập con. Tập kiểm định này chỉ dùng để kiểm thử ngược các mô hình.

In [ ]:
subset_validation = BTC_Data[1300:]
y_validation = subset_validation["signal"]
X_validation = subset_validation.loc[:, BTC_Data.columns != "signal"]

In [ ]:
X_validation.shape

### **5.1 Bộ phân loại cơ sở**

Như đã bàn ở module trước, cần huấn luyện một bộ phân loại cơ sở để so sánh với các bộ phân loại thực mà ta sẽ dùng cho nhiệm vụ này. Ta dùng [DummyClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html) của `sklearn` làm bộ phân loại cơ sở và chọn `strategy=stratified`, dự báo theo phân phối nhóm của tập huấn luyện.

Ta bắt đầu bằng việc định nghĩa các thước đo đánh giá.

In [ ]:
# test options for classification
num_folds = 10
seed = 42
scoring = "accuracy"
metrics = {
    "balanced_accuracy": "Accuracy",
    "roc_auc": "AUC",
    "neg_log_loss": "Log Loss",
    "f1_weighted": "F1",
    "precision_weighted": "Precision",
    "recall_weighted": "Recall",
}

Ta định nghĩa một hàm kiểm định chéo giúp đánh giá các thuật toán khác nhau.

In [ ]:
def run_cv(clf, X=X, y=y, metrics=metrics, cv=10, fit_params=None, n_jobs=-1):
    start = time()
    scores = cross_validate(
        estimator=clf,
        X=X,
        y=y,
        scoring=list(metrics.keys()),
        cv=cv,
        return_train_score=True,
        n_jobs=n_jobs,
        verbose=1,
        fit_params=fit_params,
    )
    duration = time() - start
    return scores, duration

Ta cũng định nghĩa một hàm giúp xử lý đầu ra của kiểm định chéo và vẽ biểu đồ, như sẽ thấy trong các ví dụ của bài này.

In [ ]:
def stack_results(scores):
    columns = pd.MultiIndex.from_tuples(
        [tuple(m.split("_", 1)) for m in scores.keys()], names=["Dataset", "Metric"]
    )
    data = np.array(list(scores.values())).T
    df = pd.DataFrame(data=data, columns=columns).iloc[:, 2:]
    results = pd.melt(df, value_name="Value")
    results.Metric = results.Metric.apply(lambda x: metrics.get(x))
    results.Dataset = results.Dataset.str.capitalize()
    return results

Cuối cùng, ta định nghĩa một hàm vẽ biểu đồ hiển thị các thước đo của mô hình dưới dạng biểu đồ hộp, để trực quan hóa phân phối hiệu suất mô hình.

In [ ]:
def plot_result(df, model=None, fname=None):
    m = list(metrics.values())
    g = sns.catplot(
        x="Dataset",
        y="Value",
        hue="Dataset",
        col="Metric",
        data=df,
        col_order=m,
        order=["Train", "Test"],
        kind="box",
        col_wrap=3,
        sharey=False,
        height=4,
        aspect=1.2,
    )
    df = df.groupby(["Metric", "Dataset"]).Value.mean().unstack().loc[m]
    for i, ax in enumerate(g.axes.flat):
        s = f"Train: {df.loc[m[i], 'Train']:>7.4f}\nTest:  {df.loc[m[i], 'Test'] :>7.4f}"
        ax.text(
            0.05,
            0.85,
            s,
            fontsize=10,
            transform=ax.transAxes,
            bbox=dict(facecolor="white", edgecolor="grey", boxstyle="round,pad=0.5"),
        )
    g.fig.suptitle(model, fontsize=16)
    g.fig.subplots_adjust(top=0.9)
    if fname:
        g.savefig(fname, dpi=300);

Bây giờ ta bắt đầu bằng việc định nghĩa bộ phân loại giả như bên dưới.

In [ ]:
dummy_clf = DummyClassifier(strategy="stratified", random_state=42)
algo = "dummy_clf"

Ta chạy bộ phân loại giả bằng kiểm định chéo với các thước đo đã định nghĩa để đo hiệu suất mô hình.

In [ ]:
fname = f"{algo}.joblib"

dummy_cv_result, run_time_algo = run_cv(dummy_clf)
joblib.dump(dummy_cv_result, fname)

Bây giờ ta có thể tạo bảng hiệu suất mô hình theo từng thước đo như bên dưới.

In [ ]:
dummy_result = stack_results(dummy_cv_result)
dummy_result.groupby(["Metric", "Dataset"]).Value.mean().unstack()

Rõ ràng hiệu suất mô hình chỉ như đoán mò: mô hình đúng khoảng một nửa số lần và các thước đo cho thấy một mô hình rất ngây thơ, đúng như ta mong đợi. Hiệu suất này sẽ là cơ sở, và khi xây dựng mô hình, ta cố gắng có một mô hình vượt trội bộ phân loại giả. Trong ô tiếp theo, ta trực quan hóa các thước đo trên bằng biểu đồ hộp.

In [ ]:
plot_result(dummy_result, model="Dummy Classifier")
plt.suptitle(
    "Fig. 5: Dummy Classifier Metrics", fontweight="bold", horizontalalignment="right"
)
plt.show()

Bây giờ ta có thể bắt đầu tìm các mô hình tốt hơn để vượt qua bộ phân loại giả.

### **5.2 Định nghĩa các thuật toán phân loại**

Ta bắt đầu bằng việc định nghĩa vài mô hình sẽ được đánh giá, rồi chọn một trong số đó.

In [ ]:
# spot check the algorithms
models = []
models.append(("LR", LogisticRegression(n_jobs=-1)))
models.append(("LDA", LinearDiscriminantAnalysis()))
models.append(("KNN", KNeighborsClassifier()))
models.append(("CART", DecisionTreeClassifier()))
models.append(("NB", GaussianNB()))
# Neural Network
models.append(("NN", MLPClassifier()))
# Ensemble Models
# Boosting methods
models.append(("AB", AdaBoostClassifier()))
models.append(("GBM", GradientBoostingClassifier()))
# Bagging methods
models.append(("RF", RandomForestClassifier(n_jobs=-1)))

Bây giờ ta đánh giá hiệu suất các mô hình này bằng kiểm định chéo `kFold` và ghi lại trung bình và phương sai.

In [ ]:
results = []
names = []
for name, model in models:
    kfold = KFold(n_splits=num_folds)
    cv_results = cross_val_score(model, X_train, y_train, cv=kfold, scoring=scoring)
    results.append(cv_results)
    names.append(name)
    msg = "%s: %f (%f)" % (name, cv_results.mean(), cv_results.std())
    print(msg)

Nhìn vào các kết quả đã ghi ở trên, bộ phân loại rừng ngẫu nhiên vượt trội các mô hình khác, và ta chọn nó làm mô hình ưa thích. Ta cũng có thể trực quan hóa hiệu suất các thuật toán bằng biểu đồ hộp như bên dưới.

In [ ]:
# compare algorithms
fig = plt.figure()
ax = fig.add_subplot(111)
plt.boxplot(results)
ax.set_xticklabels(names)
fig.set_size_inches(15, 8)
plt.suptitle(
    "Fig. 6: Algorithm Comparison", fontweight="bold", horizontalalignment="right"
)
plt.show()

## **6. Tinh chỉnh siêu tham số**

Vì rừng ngẫu nhiên là mô hình tốt nhất, ta tìm cách tinh chỉnh siêu tham số của nó để cải thiện hiệu suất hơn nữa.

### **6.1 Tìm kiếm lưới**

In [ ]:
# Grid Search: Random Forest Classifier

scaler = StandardScaler().fit(X_train)
rescaledX = scaler.transform(X_train)
n_estimators = [20, 80]
max_depth = [5, 10]
criterion = ["gini", "entropy"]
param_grid = dict(n_estimators=n_estimators, max_depth=max_depth, criterion=criterion)
model = RandomForestClassifier(n_jobs=-1)
kfold = KFold(n_splits=num_folds)
grid = GridSearchCV(estimator=model, param_grid=param_grid, scoring=scoring, cv=kfold)
grid_result = grid.fit(rescaledX, y_train)

# Print Results
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))
means = grid_result.cv_results_["mean_test_score"]
stds = grid_result.cv_results_["std_test_score"]
params = grid_result.cv_results_["params"]
ranks = grid_result.cv_results_["rank_test_score"]
for mean, stdev, param, rank in zip(means, stds, params, ranks):
    print("#%d %f (%f) with: %r" % (rank, mean, stdev, param))

Bây giờ ta có thể gán các siêu tham số tốt nhất cho mô hình và dùng nó để so sánh với bộ phân loại giả. Ta lặp lại các bước như ở bài toán bộ phân loại giả.

In [ ]:
# prepare model
model = RandomForestClassifier(
    criterion="gini", n_estimators=80, max_depth=10, n_jobs=-1
)  # `rbf` is default kernel

In [ ]:
algo_forest = "random_forest"

In [ ]:
fname = f"{algo_forest}.joblib"

random_forest_cv_result, run_time_algo_forest = run_cv(model)
joblib.dump(random_forest_cv_result, fname)

Bây giờ ta có thể hiển thị bảng hiệu suất mô hình bên dưới.

In [ ]:
rf_result = stack_results(random_forest_cv_result)
rf_result.groupby(["Metric", "Dataset"]).Value.mean().unstack()

Ô bên dưới hiển thị các thước đo trên dưới dạng biểu đồ hộp.

In [ ]:
plot_result(rf_result, model="Random Forest Classifier")
plt.suptitle(
    "Fig. 7: Random Forest Classifier Metrics",
    fontweight="bold",
    horizontalalignment="right",
)
plt.show()

Rõ ràng bộ phân loại rừng ngẫu nhiên vượt xa bộ phân loại giả, nhưng có vẻ ta gặp vấn đề quá khớp. Trong nhiệm vụ dự báo, ta muốn tìm cách giảm thiểu điều này. Bảng bên dưới so sánh bộ phân loại rừng ngẫu nhiên và bộ phân loại giả.

In [ ]:
results = {
    "Baseline": dummy_result,
    "Random Forest": rf_result,
}
df = pd.DataFrame()
for model, result in results.items():
    df = pd.concat(
        [
            df,
            result.groupby(["Metric", "Dataset"])
            .Value.mean()
            .unstack()["Test"]
            .to_frame(model),
        ],
        axis=1,
    )

df.T.sort_values("AUC", ascending=False)

Đã chọn rừng ngẫu nhiên, bây giờ ta có thể huấn luyện nó trên dữ liệu và phân tích hiệu suất trên tập dữ liệu giữ lại.

In [ ]:
model = RandomForestClassifier(
    criterion="gini", n_estimators=80, max_depth=10, n_jobs=-1
)
model.fit(X_train, y_train)
# estimate accuracy on validation set
predictions = model.predict(X_test)
print(accuracy_score(y_test, predictions))
print(confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions))

Mô hình cho thấy tiềm năng dự báo tốt hơn trên tập dữ liệu giữ lại, với hiệu suất trên $94\%$.

Để đánh giá hiệu suất mô hình, ta có thể vẽ ma trận nhầm lẫn, cho phép trực quan hóa độ nhạy (recall), độ đặc hiệu, độ chính xác (accuracy) và độ chuẩn xác (precision) của mô hình.

In [ ]:
df_cm = pd.DataFrame(
    confusion_matrix(y_test, predictions),
    columns=np.unique(y_test),
    index=np.unique(y_test),
)
df_cm.index.name = "Actual"
df_cm.columns.name = "Predicted"
sns.heatmap(df_cm, cmap="Blues", annot=True, annot_kws={"size": 16})  # font sizes
plt.suptitle("Fig. 8: Confusion Matrix", fontweight="bold", horizontalalignment="right")
plt.show()

Ta xác định các biến có sức dự báo mạnh hơn như bên dưới.

In [ ]:
Importance = pd.DataFrame(
    {"Importance": model.feature_importances_ * 100}, index=X.columns
)
Importance.sort_values("Importance", axis=0, ascending=True).plot(
    kind="barh", color="r", figsize=(12, 8)
)
plt.xlabel("Variable Importance")
plt.suptitle(
    "Fig. 9: Variable Importance", fontweight="bold", horizontalalignment="right"
)
plt.show()

### **6.2 Tối ưu hóa Bayes**

Quy trình tối ưu hóa Bayes dùng quá trình Gauss để mô hình hóa trung bình và phương sai của hàm mục tiêu. Ở mỗi vòng lặp, nó thử các tổ hợp giá trị siêu tham số khác nhau và xây dựng mô hình mới bằng thông tin trong quá khứ của mô hình. Ưu điểm là giảm thời gian huấn luyện để đạt kết quả tốt hơn.

Ta bắt đầu bằng việc định nghĩa hàm `kFold` phân tầng dùng để kiểm định chéo trên tập huấn luyện.

In [ ]:
def stratified_kfold_score(clf, X_train, y_train, n_fold):
    X, y = X_train.values, y_train.values
    strat_kfold = StratifiedKFold(n_splits=n_fold, shuffle=True, random_state=1)
    accuracy_list = []

    for train_index, test_index in strat_kfold.split(X, y):
        x_train_fold, x_test_fold = X[train_index], X[test_index]
        y_train_fold, y_test_fold = y[train_index], y[test_index]
        clf.fit(x_train_fold, y_train_fold)
        preds = clf.predict(x_test_fold)
        accuracy_test = accuracy_score(preds, y_test_fold)
        accuracy_list.append(accuracy_test)

    return np.array(accuracy_list).mean()

Ta cũng định nghĩa một hàm chứa các siêu tham số của mô hình mà ta muốn tinh chỉnh, gồm `max_samples`, `max_features` và `n_estimators`.

In [ ]:
def bo_params_rf(max_samples, n_estimators, max_features):
    params = {
        "max_samples": max_samples,
        "max_features": max_features,
        "n_estimators": int(n_estimators),
    }
    clf = RandomForestClassifier(
        max_samples=params["max_samples"],
        max_features=params["max_features"],
        n_estimators=params["n_estimators"],
    )
    score = stratified_kfold_score(clf, X_train, y_train, 5)
    return score

Trong ô tiếp theo, ta cho khoảng không gian siêu tham số mà tối ưu hóa Bayes sẽ tìm kiếm.

In [ ]:
rf_bo = BayesianOptimization(
    bo_params_rf,
    {"max_samples": (0.2, 1), "max_features": (0.5, 1), "n_estimators": (50, 200)},
)

Bây giờ ta có thể áp dụng tối ưu hóa Bayes để tìm kiếm trong không gian siêu tham số và xuất kết quả ở mỗi vòng lặp. Ô bên dưới xuất hiệu suất cho từng tổ hợp siêu tham số mà nó tìm được và đánh dấu màu xanh tổ hợp có điểm cao hơn các tổ hợp mà nó đã chọn trước đó.

In [ ]:
results = rf_bo.maximize(n_iter=200, init_points=20, acq="ei")

Bây giờ ta có thể hiển thị các siêu tham số tốt nhất mà tối ưu hóa Bayes đã tìm thấy.

In [ ]:
params = rf_bo.max["params"]
params["n_estimators"] = int(params["n_estimators"])
print(params)

Bây giờ ta có thể gán các siêu tham số này cho mô hình như bên dưới.

In [ ]:
rf_v1 = RandomForestClassifier(
    max_samples=params["max_samples"],
    max_features=params["max_features"],
    n_estimators=params["n_estimators"],
)

Bây giờ ta dùng tập huấn luyện để huấn luyện mô hình trên với các siêu tham số tốt nhất đã chọn.

In [ ]:
rf_v1.fit(X_train, y_train)

Từ mô hình vừa huấn luyện, ta dùng nó để dự báo trên tập dữ liệu giữ lại và đánh giá hiệu suất mô hình bằng báo cáo phân loại.

In [ ]:
preds = rf_v1.predict(X_test)

In [ ]:
print(classification_report(preds, y_test))

Báo cáo phân loại ở trên cho thấy mô hình dự báo đúng các tín hiệu với độ chuẩn xác cao. Ta có thể in ra các giá trị precision, recall và f1 để hiểu rõ hơn báo cáo phân loại.

In [ ]:
print("Precision:{}".format(precision_score(preds, y_test)))
print("Recall:{}".format(recall_score(preds, y_test)))
print("F1 Score:{}".format((f1_score(preds, y_test))))

Như đã nói, ta đạt hiệu suất cao trên báo cáo phân loại. Liệu có thể cải thiện hiệu suất mô hình không? Ta thử cài đặt kiểm định `kFold` phân tầng bên dưới.

In [ ]:
cv = StratifiedKFold(n_splits=5, random_state=123, shuffle=True)
scores = []

for (train, test), i in zip(cv.split(X_train, y_train), range(5)):
    rf_v1.fit(X.iloc[train], y.iloc[train])
    preds = rf_v1.predict(X_train.iloc[test])
    accuracy = accuracy_score(preds, y_train.iloc[test])
    scores.append(accuracy)

df_val = pd.DataFrame(scores, columns=["Accuracy Test"])
print(
    "KFold validation mean accuracy on test set : {}".format(
        df_val["Accuracy Test"].mean()
    )
)

Độ chính xác trên tập dữ liệu kiểm tra cho kết quả tương đương với những gì ta đã thấy cho đến giờ.

### **7. Kiểm thử ngược (Backtesting)**

Bây giờ ta muốn xem chiến lược giao dịch sẽ hoạt động ra sao trên dữ liệu lịch sử, và điều này khả thi nhờ kiểm thử ngược. Kiểm thử ngược cho phép ta mô phỏng kết quả mình sẽ đạt được trong quá khứ mà không phải dùng đồng vốn nào.

In [ ]:
# Create column for Strategy Returns by multiplying the daily returns by the position that was held at close
# of business the previous day
backtestdata = pd.DataFrame(index=X_validation.index)
validation_prediction = rf_v1.predict(X_validation)
backtestdata["signal_pred"] = validation_prediction
backtestdata["signal_actual"] = y_validation
backtestdata["Market Returns"] = X_validation["Close"].pct_change()
backtestdata["Actual Returns"] = backtestdata["Market Returns"] * backtestdata[
    "signal_actual"
].shift(1)
backtestdata["Strategy Returns"] = backtestdata["Market Returns"] * backtestdata[
    "signal_pred"
].shift(1)
# backtestdata=backtestdata.reset_index()
backtestdata.head()

fig, axs = plt.subplots(2, figsize=(15, 15))
fig.suptitle("Vertically stacked subplots")
axs[0].plot(backtestdata["Strategy Returns"])
axs[0].set_title("Strategy Returns")
axs[1].plot(backtestdata["Actual Returns"])
axs[1].set_title("Bitcoin Daily Returns")

for ax in axs.flat:
    ax.set(xlabel="Predicted Days", ylabel="Returns")

# `backtestdata`[['Strategy Returns','Actual Returns']].plot()
# `plt.title`('Cumulative Strategy Returns vs Cumulative Actual Returns')
plt.suptitle(
    "Fig. 10: Strategy and Bitcoin Daily Returns Separate Graphs",
    fontweight="bold",
    horizontalalignment="right",
)
plt.show()

Ở các biểu đồ trên, ta trực quan hóa lợi suất chiến lược và lợi suất ngày của Bitcoin trong hai hình riêng. Bên dưới, ta đặt các hình trên vào cùng một hình.

In [ ]:
backtestdata[["Strategy Returns", "Actual Returns"]].plot(figsize=(15, 8))
plt.title("Strategy Returns vs BITCOIN DAILY Returns chart 2")
plt.suptitle(
    "Fig. 11: Strategy and Bitcoin Daily Returns",
    fontweight="bold",
    horizontalalignment="right",
)
plt.xlabel("Test Days")
plt.ylabel("Returns")
plt.show()

Ta có thể thấy rõ lợi suất của chiến lược và lợi suất ngày của Bitcoin giống nhau ở hầu hết các ngày kiểm tra, trừ rất ít ngày. Bây giờ ta có thể vẽ lợi suất tích lũy của cả chiến lược và Bitcoin để so sánh kết quả ta kỳ vọng trong tương lai gần nếu áp dụng mô hình giao dịch này.

In [ ]:
backtestdata[["Strategy Returns", "Actual Returns"]].cumsum().plot(figsize=(15, 8))
plt.xlabel("Test Days")
plt.ylabel("Cumulative Returns")
plt.suptitle(
    "Fig. 12: Cumulative Strategy Returns vs Cumulative BITCOIN  Returns",
    fontweight="bold",
    horizontalalignment="right",
)
plt.show()

## **7. Kết luận**

Trong bài này, ta đã học cách áp dụng tối ưu hóa Bayes và tìm kiếm lưới lên mô hình để cải thiện hiệu suất. Ta cũng thấy cách dùng các mô hình cơ sở làm tham chiếu cho các mô hình dùng để dự báo tín hiệu mua hoặc bán. Ta dự đoán còn có thể làm nhiều hơn để cải thiện mô hình, và khuyến khích người học tìm hiểu thêm.

**Tài liệu tham khảo**

1. Snoek, Jasper, et al. "Practical Bayesian Optimization of Machine Learning Algorithms." *Advances in Neural Information Processing Systems*, vol. 25, 2012.
2. Shalini, Talwar, Shah Pranav, and Shah Utkarsh. "Picking buy-sell signals: A practitioner's perspective on key technical indicators for selected Indian firms." Studies in Business & Economics 14.3 (2019).
3. Bergstra, James, et al. "Algorithms for Hyper-parameter Optimization." *Advances in Neural Information Processing Systems*, vol. 24, 2011. https://proceedings.neurips.cc/paper/2011/file/86e8f7ab32cfd12577bc2619bc635690-Paper.pdf